In [3]:
import pandas as pd

df = pd.read_csv("C:\\Users\\Ritik Chauhan\\OneDrive\\Desktop\\pythonp\\Customer Churn Analysis\\data\\cleaneddataset.csv")
print(df.shape)
print(df.dtypes)

(7043, 24)
customerID           object
gender               object
SeniorCitizen         int64
Partner              object
Dependents           object
tenure                int64
PhoneService         object
MultipleLines        object
InternetService      object
OnlineSecurity       object
OnlineBackup         object
DeviceProtection     object
TechSupport          object
StreamingTV          object
StreamingMovies      object
Contract             object
PaperlessBilling     object
PaymentMethod        object
MonthlyCharges      float64
TotalCharges        float64
Churn                object
ChurnFlag             int64
TenureBucket         object
NumServices           int64
dtype: object


In [4]:
# Separate target from features
drop_cols = ['customerID', 'Churn', 'ChurnFlag', 'TenureBucket']
X = df.drop(columns=drop_cols)
y = df['ChurnFlag']

# One-hot encode text columns
X = pd.get_dummies(X, drop_first=True)

print(X.shape)
print(y.shape)

(7043, 31)
(7043,)


In [5]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(X_train.shape)
print(X_test.shape)

(5634, 31)
(1409, 31)


In [6]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## Logistic Regression Model

Using logistic regression since it's interpretable — the coefficients
directly show which features increase or decrease churn risk, which is valuable for
a business audience who need to understand *why*, not just *what*.

In [7]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000, class_weight='balanced')
model.fit(X_train_scaled, y_train)

print("Model trained successfully!")

Model trained successfully!


In [8]:
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix

y_pred = model.predict(X_test_scaled)
y_prob = model.predict_proba(X_test_scaled)[:, 1]

print("AUC-ROC:", round(roc_auc_score(y_test, y_prob), 3))
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))

AUC-ROC: 0.841

Classification Report:
               precision    recall  f1-score   support

           0       0.90      0.72      0.80      1035
           1       0.51      0.78      0.61       374

    accuracy                           0.74      1409
   macro avg       0.70      0.75      0.71      1409
weighted avg       0.80      0.74      0.75      1409

Confusion Matrix:
 [[748 287]
 [ 81 293]]


In [9]:
import pandas as pd

coefficients = pd.Series(model.coef_[0], index=X.columns).sort_values()

print("Top 5 factors that INCREASE churn risk:")
print(coefficients.tail(5)[::-1])

print("\nTop 5 factors that DECREASE churn risk:")
print(coefficients.head(5))

Top 5 factors that INCREASE churn risk:
InternetService_Fiber optic    0.812562
TotalCharges                   0.499089
StreamingMovies_Yes            0.264877
StreamingTV_Yes                0.252148
MultipleLines_Yes              0.202123
dtype: float64

Top 5 factors that DECREASE churn risk:
tenure               -1.163832
MonthlyCharges       -1.047362
Contract_Two year    -0.617225
Contract_One year    -0.296343
OnlineSecurity_Yes   -0.134164
dtype: float64


### Observation:

`MonthlyCharges` shows a *negative* coefficient here, meaning the model treats higher
monthly spend as reducing churn risk. This contradicts the SQL analysis, where
higher-paying fiber customers churned *more*, not less.

This is worth investigating rather than ignoring — a likely cause is
**multicollinearity**, since `tenure`, `MonthlyCharges`, and `TotalCharges` are
mathematically related (`TotalCharges ≈ tenure × MonthlyCharges`).

### Testing for multicollinearity (VIF check)

Variance Inflation Factor (VIF) measures how much a feature can be predicted from
the other features. VIF > 5 suggests meaningful multicollinearity; VIF > 10 is
considered severe.

In [12]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

numeric_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']
vif_data = pd.DataFrame()
vif_data['feature'] = numeric_cols
vif_data['VIF'] = [variance_inflation_factor(X[numeric_cols].values, i) for i in range(len(numeric_cols))]
print(vif_data)

          feature       VIF
0          tenure  6.332328
1  MonthlyCharges  3.355660
2    TotalCharges  8.075070


VIF for `TotalCharges` came out at 8+, confirming meaningful multicollinearity
between the three charge/tenure-related features. Since `TotalCharges` is largely
redundant with `tenure` and `MonthlyCharges`, dropping it and retraining the model.

In [13]:
# Drop the redundant feature
X_fixed = X.drop(columns=['TotalCharges'])

X_train2, X_test2, y_train2, y_test2 = train_test_split(
    X_fixed, y, test_size=0.2, random_state=42, stratify=y
)

scaler2 = StandardScaler()
X_train2_scaled = scaler2.fit_transform(X_train2)
X_test2_scaled = scaler2.transform(X_test2)

model2 = LogisticRegression(max_iter=1000, class_weight='balanced')
model2.fit(X_train2_scaled, y_train2)

y_pred2 = model2.predict(X_test2_scaled)
y_prob2 = model2.predict_proba(X_test2_scaled)[:, 1]

print("New AUC-ROC:", round(roc_auc_score(y_test2, y_prob2), 3))

coefficients2 = pd.Series(model2.coef_[0], index=X_fixed.columns).sort_values()
print("\nTop 5 factors that INCREASE churn risk:")
print(coefficients2.tail(5)[::-1])
print("\nTop 5 factors that DECREASE churn risk:")
print(coefficients2.head(5))

New AUC-ROC: 0.838

Top 5 factors that INCREASE churn risk:
InternetService_Fiber optic       0.834962
StreamingMovies_Yes               0.276069
StreamingTV_Yes                   0.264452
MultipleLines_Yes                 0.215072
PaymentMethod_Electronic check    0.189002
dtype: float64

Top 5 factors that DECREASE churn risk:
MonthlyCharges       -0.941350
tenure               -0.754268
Contract_Two year    -0.614324
Contract_One year    -0.297756
OnlineSecurity_Yes   -0.129124
dtype: float64


### Problem Still Persists

Removing `TotalCharges` didn't fully resolve the issue — `MonthlyCharges` is still
negative. Looking at the updated top features (Fiber optic, StreamingMovies,
StreamingTV, MultipleLines all near the top), the remaining multicollinearity is
likely spread across the various service/add-on features that themselves drive up
`MonthlyCharges`.

Rather than keep removing features and losing information, validating the driver
rankings using a Random Forest model that isn't affected by this issue.

## Random Forest Model (Validation)

Random Forest doesn't rely on linear coefficients, so it isn't affected by
multicollinearity the same way — it produces a clean feature importance ranking,
useful here as an independent check on the logistic regression's driver rankings.

In [14]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=200, random_state=42, class_weight='balanced')
rf.fit(X_train2, y_train2)  # tree models don't need scaling

rf_importance = pd.Series(rf.feature_importances_, index=X_fixed.columns).sort_values(ascending=False)
print("Top 10 churn drivers according to Random Forest:")
print(rf_importance.head(10))

from sklearn.metrics import roc_auc_score
rf_prob = rf.predict_proba(X_test2)[:, 1]
print("\nRandom Forest AUC:", round(roc_auc_score(y_test2, rf_prob), 3))

Top 10 churn drivers according to Random Forest:
tenure                            0.217280
MonthlyCharges                    0.192755
Contract_Two year                 0.064170
NumServices                       0.045482
InternetService_Fiber optic       0.039385
PaymentMethod_Electronic check    0.038906
Contract_One year                 0.035226
gender_Male                       0.030058
OnlineSecurity_Yes                0.029423
PaperlessBilling_Yes              0.027174
dtype: float64

Random Forest AUC: 0.819


## Conclusion

Both models independently agree on the top churn drivers: **tenure, MonthlyCharges,
and two-year contracts**. Agreement across two different model types (linear and
tree-based) gives confidence these are genuine patterns in the data, not artifacts
of how one particular model handles correlated features.

**Final churn drivers for the business recommendation:**
- Short tenure (especially year 1) → highest churn risk
- Higher monthly spend → correlates with churn, likely tied to fiber/streaming bundles
- Month-to-month contracts → far higher churn than 1-year/2-year contracts